# Notebook 00 — Environment and Data Download

**Task ID:** `ENV-001`  
**Phase:** Phase 0 — Environment Setup & Data Ingestion  
**Purpose:** Establish a reproducible Colab / Local environment, configure dependencies, and download all datasets without exposing credentials.

---

## 1. Project Objective and Dataset List

### Datasets Used in ClaimVision AI:
1. **Fraud Dataset:** `vinayjose/car-damage-dataset` (Kaggle)
   - Used for visual fraud-risk classification (Phase 1 & 2).
2. **Severity Dataset:** `anujms/car-damage-severity-dataset` (Kaggle)
   - 3-class damage severity: Minor, Moderate, Severe (Phase 5–9).
3. **Detection Dataset:** COCO Car Damage Detection Dataset
   - Damage localization and 5 damaged-part classes (Phase 10–12).

## 2. Universal Colab & Local Environment Bootstrap

In [33]:
import os
import sys
import random
from pathlib import Path

# --- Colab / Local Universal Sync ---
if 'google.colab' in sys.modules or os.path.exists('/content'):
    repo_dir = Path('/content/NPN-Car-Insurance')
    import subprocess
    if not (repo_dir / '.git').exists():
        print('🚀 New Colab session detected. Cloning repository...')
        subprocess.run(['git', 'clone', 'https://github.com/AmitavaDatta2004/NPN-Car-Insurance.git', str(repo_dir)], check=True)
    else:
        print('🔄 Colab repository exists. Syncing with latest GitHub commits...')
        # Use fetch + reset --hard so any temporary Colab autosaves never cause merge conflicts
        subprocess.run(['git', '-C', str(repo_dir), 'fetch', 'origin', 'main'], check=True)
        subprocess.run(['git', '-C', str(repo_dir), 'reset', '--hard', 'origin/main'], check=True)
    
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'imagehash', 'kagglehub'], check=False)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(repo_dir / 'ml')], check=False)
    
    ml_src = str(repo_dir / 'ml' / 'src')
    if ml_src not in sys.path:
        sys.path.insert(0, ml_src)
    os.chdir(str(repo_dir))
    print('✅ Working directory updated to:', os.getcwd())
else:
    # Local fallback for running directly on your laptop
    for candidate in [Path('ml/src').resolve(), Path('../ml/src').resolve(), Path('../../ml/src').resolve()]:
        if candidate.exists() and str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
            break
    print('✅ Running locally. Working directory:', os.getcwd())



🔄 Colab repository exists. Syncing with latest GitHub commits...
✅ Working directory updated to: /content/NPN-Car-Insurance


## 3. GPU and System Hardware Verification

In [34]:
import platform
import torch

print('=' * 50)
print(f'OS / Platform      : {platform.platform()}')
print(f'Python Version     : {platform.python_version()}')
print(f'PyTorch Version    : {torch.__version__}')
print(f'CUDA Available     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU Device Name    : {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory (VRAM)  : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB')
else:
    print('GPU Device Name    : CPU (No CUDA device found)')
print('=' * 50)


OS / Platform      : Linux-6.6.122+-x86_64-with-glibc2.39
Python Version     : 3.13.15
PyTorch Version    : 2.11.0+cu128
CUDA Available     : True
GPU Device Name    : Tesla T4
GPU Memory (VRAM)  : 14.56 GB


## 4. Reproducible Random Seed Initialization

In [35]:
import numpy as np

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
    print(f'🔒 Global scientific reproducibility seed locked to: {seed}')

set_seed(42)


🔒 Global scientific reproducibility seed locked to: 42


## 5. Download Fraud Dataset (`vinayjose/car-damage-dataset`)

In [36]:
import kagglehub
import shutil

raw_dir = Path('data/raw').resolve()
raw_dir.mkdir(parents=True, exist_ok=True)
os.environ['KAGGLEHUB_CACHE'] = str(raw_dir)

fraud_dir = raw_dir / 'vinayjose_car_damage'

# Check if dataset is already downloaded and non-empty to avoid re-downloading
if fraud_dir.exists() and any(fraud_dir.iterdir()):
    print(f'✅ Fraud dataset already present at: {fraud_dir} (Skipping download)')
    fraud_path = str(fraud_dir)
else:
    print(f'Downloading fraud dataset to: {fraud_dir}...')
    try:
        fraud_path = kagglehub.dataset_download('vinayjose/car-damage-dataset', output_dir=str(fraud_dir))
        print(f'✅ Fraud dataset successfully downloaded to: {fraud_path}')
        
        # If kagglehub placed into /kaggle/input or another cache path, symlink or copy to fraud_dir
        downloaded_dir = Path(fraud_path)
        if downloaded_dir.resolve() != fraud_dir.resolve() and downloaded_dir.exists():
            if not (fraud_dir.exists() and any(fraud_dir.iterdir())):
                print(f'🔗 Linking {downloaded_dir} -> {fraud_dir}...')
                try:
                    fraud_dir.symlink_to(downloaded_dir, target_is_directory=True)
                    print('✅ Symlinked successfully.')
                except Exception:
                    print('Copying files into data/raw/vinayjose_car_damage...')
                    shutil.copytree(downloaded_dir, fraud_dir, dirs_exist_ok=True)
                    print('✅ Copied successfully.')
    except Exception as e:
        print(f'⚠️ Kagglehub download note: {e}')
        fraud_path = str(fraud_dir)


✅ Fraud dataset already present at: /content/NPN-Car-Insurance/data/raw/vinayjose_car_damage (Skipping download)


## 6. Download Severity Dataset (`anujms/car-damage-severity-dataset`)

In [37]:
severity_dir = raw_dir / 'car_damage_severity'

# Check if dataset is already downloaded and non-empty to avoid re-downloading
if severity_dir.exists() and any(severity_dir.iterdir()):
    print(f'✅ Severity dataset already present at: {severity_dir} (Skipping download)')
    sev_path = str(severity_dir)
else:
    print(f'Downloading severity dataset to: {severity_dir}...')
    try:
        # Use the public prajwalbhamere/car-damage-severity-dataset (Minor, Moderate, Severe)
        sev_path = kagglehub.dataset_download('prajwalbhamere/car-damage-severity-dataset', output_dir=str(severity_dir))
        print(f'✅ Severity dataset successfully downloaded to: {sev_path}')
        
        # If kagglehub placed into /kaggle/input or another cache path, symlink or copy to severity_dir
        downloaded_dir = Path(sev_path)
        if downloaded_dir.resolve() != severity_dir.resolve() and downloaded_dir.exists():
            if not (severity_dir.exists() and any(severity_dir.iterdir())):
                print(f'🔗 Linking {downloaded_dir} -> {severity_dir}...')
                try:
                    severity_dir.symlink_to(downloaded_dir, target_is_directory=True)
                    print('✅ Symlinked successfully.')
                except Exception:
                    print('Copying files into data/raw/car_damage_severity...')
                    shutil.copytree(downloaded_dir, severity_dir, dirs_exist_ok=True)
                    print('✅ Copied successfully.')
    except Exception as e:
        print(f'⚠️ Kagglehub download note: {e}')
        sev_path = str(severity_dir)


✅ Severity dataset already present at: /content/NPN-Car-Insurance/data/raw/car_damage_severity (Skipping download)


## 7. Download COCO Detection Dataset (`lplenka/cococar-damage-detection-dataset`)

Used for generic damage localization and 5 damaged-part classes (Phase 8–10, Notebooks 10–12).
Contains 59 train, 11 validation, and 8 test images with COCO-format bounding boxes and segmentation.

In [38]:
coco_dir = raw_dir / 'coco_car_damage'

# Check if dataset is already downloaded and non-empty to avoid re-downloading
if coco_dir.exists() and any(coco_dir.iterdir()):
    print(f'✅ COCO Detection dataset already present at: {coco_dir} (Skipping download)')
    coco_path = str(coco_dir)
else:
    print(f'Downloading COCO Detection dataset to: {coco_dir}...')
    try:
        # Public unauthenticated mirror: kunalmadaan03/cococardamagedetectiondataset
        # Contains the exact 59 train, 11 val images with damage and 5 car parts categories
        coco_path = kagglehub.dataset_download('kunalmadaan03/cococardamagedetectiondataset', output_dir=str(coco_dir))
        print(f'✅ COCO Detection dataset successfully downloaded to: {coco_path}')
        
        # If kagglehub placed into cache path, symlink or copy to coco_dir
        downloaded_dir = Path(coco_path)
        if downloaded_dir.resolve() != coco_dir.resolve() and downloaded_dir.exists():
            if not (coco_dir.exists() and any(coco_dir.iterdir())):
                print(f'🔗 Linking {downloaded_dir} -> {coco_dir}...')
                try:
                    coco_dir.symlink_to(downloaded_dir, target_is_directory=True)
                    print('✅ Symlinked successfully.')
                except Exception:
                    import shutil
                    print('Copying files into data/raw/coco_car_damage...')
                    shutil.copytree(downloaded_dir, coco_dir, dirs_exist_ok=True)
                    print('✅ Copied successfully.')
    except Exception as e:
        print(f'⚠️ Kagglehub download note: {e}')
        coco_path = str(coco_dir)


100%|██████████| 14.7M/14.7M [00:01<00:00, 8.91MB/s]

Extracting files...


✅ COCO Detection dataset successfully downloaded to: /content/NPN-Car-Insurance/data/raw/coco_car_damage


## 8. Dataset Directory Structure & Registry Creation


In [39]:
import json
import hashlib

registry = {
    'fraud_dataset': {
        'source': 'vinayjose/car-damage-dataset',
        'path': str(fraud_dir),
        'exists': fraud_dir.exists(),
    },
    'severity_dataset': {
        'source': 'prajwalbhamere/car-damage-severity-dataset',
        'path': str(severity_dir),
        'exists': severity_dir.exists(),
    },
    'coco_detection_dataset': {
        'source': 'kunalmadaan03/cococardamagedetectiondataset',
        'path': str(coco_dir),
        'exists': coco_dir.exists(),
    }
}

registry_path = Path('data/dataset_registry.json')
registry_path.parent.mkdir(parents=True, exist_ok=True)
with open(registry_path, 'w') as f:
    json.dump(registry, f, indent=2)

print(f'✅ Dataset registry saved to: {registry_path}')
print(json.dumps(registry, indent=2))


✅ Dataset registry saved to: data/dataset_registry.json
{
  "fraud_dataset": {
    "source": "vinayjose/car-damage-dataset",
    "path": "/content/NPN-Car-Insurance/data/raw/vinayjose_car_damage",
    "exists": true
  },
  "severity_dataset": {
    "source": "prajwalbhamere/car-damage-severity-dataset",
    "path": "/content/NPN-Car-Insurance/data/raw/car_damage_severity",
    "exists": true
  },
  "coco_detection_dataset": {
    "source": "kunalmadaan03/cococardamagedetectiondataset",
    "path": "/content/NPN-Car-Insurance/data/raw/coco_car_damage",
    "exists": true
  }
}
